# Nemotron 3.5 ASR Streaming 0.6B: Full Fine-Tune on OpenSLR 104

This notebook fine-tunes all weights of [`nvidia/nemotron-3.5-asr-streaming-0.6b`](https://huggingface.co/nvidia/nemotron-3.5-asr-streaming-0.6b) on the OpenSLR 104 Hinglish training set (41,531 clips, 67.5 h) with NeMo, and scores the best checkpoint on the test set. It runs in Google Colab on an A100 (40 GB or 80 GB).

- **Prompt:** `hi-IN` for every training, validation and test clip (`data/notes/more.md`).
- **Recipe:** NVIDIA's fine-tune recipe ([blog](https://developer.nvidia.com/blog/fine-tuning-nvidia-nemotron-for-saudi-arabic-dialects-with-a-path-to-other-languages/)): AdamW with Noam annealing, lhotse bucketing by batch duration, bf16, all 24 encoder layers trained. The model's own config (features, SpecAugment, streaming attention) is kept; only the data, optimizer and trainer settings change.
- **Scoring:** the project `score` (strict WER and T-WER) on whole clips with the encoder's attention limited to the streaming latency: `att_context_size` [56, 6] for 560 ms and [56, 13] for 1.12 s.
- **Inputs on Drive:** the project folder, and `openslr104_train.zip`, `openslr104_validation.zip`, `openslr104_test.zip` (the clips in `data/processed/openslr104/<split>/`).
- **Outputs on Drive:** `checkpoints/nemotron-3.5-asr-streaming-0.6b/` (`last.ckpt` for resuming, `best.nemo`, `val_scores.csv`, `state.json` with GPU type and training hours), `results/nemotron-3.5-asr-streaming-0.6b-finetuned/`, and two rows in `results/summary.csv`.

**Run order.** First session: every cell. Later sessions (Colab disconnects, or a new day): Settings, Setup, Audio, Manifests, Scoring, Model, then Train, which continues from `last.ckpt`. After training stops: the Cost and Final scores cells.

## Settings

In [1]:
PROJECT = "/content/drive/MyDrive/projects/asr"
MODEL = "nvidia/nemotron-3.5-asr-streaming-0.6b"
PROMPT = "hi-IN"  # data/notes/more.md

SPLITS = ("train", "validation", "test")
ZIPS = {split: f"{PROJECT}/openslr104_{split}.zip" for split in SPLITS}
AUDIO_ROOT = "/content/openslr104"
MANIFEST_DIR = "/content/manifests"
CKPT_DIR = f"{PROJECT}/checkpoints/nemotron-3.5-asr-streaming-0.6b"
OUT_DIR = f"{PROJECT}/results/nemotron-3.5-asr-streaming-0.6b-finetuned"

# Streaming latency -> att_context_size [left, right] in 80 ms frames (model card).
CONTEXT = {560: [56, 6], 1120: [56, 13]}
VAL_MS = 560

# Optimizer (NVIDIA recipe: AdamW + Noam). Noam's lr is a multiplier: peak = lr / sqrt(d_model * warmup).
PEAK_LR = 1e-4
WARMUP_STEPS = 100
D_MODEL = 1024
OPTIM = {
    "name": "adamw",
    "lr": PEAK_LR * (D_MODEL * WARMUP_STEPS) ** 0.5,
    "betas": [0.9, 0.98],
    "weight_decay": 1e-3,
    "sched": {"name": "NoamAnnealing", "d_model": D_MODEL, "warmup_steps": WARMUP_STEPS, "min_lr": 1e-6},
}

# Batching (lhotse bucketing as in the recipe). None picks 200 s on 80 GB and 100 s on 40 GB.
# After an out-of-memory error, set a lower number here and run the Model and Train cells again.
BATCH_DURATION = None
NUM_BUCKETS = 30
QUADRATIC_DURATION = 15
NUM_WORKERS = 8

# Trainer
PRECISION = "bf16-mixed"
GRAD_CLIP = 0.5
MAX_STEPS = 20_000
VAL_EVERY = 1_000  # steps between validation runs
PATIENCE = 3  # validation runs without a lower WER before training stops
CKPT_MINUTES = 30
INFER_BATCH = 32

# Small-set check
SMALL_N = 50
SMALL_STEPS = 200
SMALL_BATCH_DURATION = 60

## Setup

NeMo comes from GitHub `main`, as the model card says (NeMo 26.06 or newer). If Colab asks to restart the session after the install, restart, then run the Settings cell again and continue from the next cell.

In [2]:
!apt-get install -y -qq libsndfile1 ffmpeg
!pip install -q Cython packaging
!pip install -q "nemo_toolkit[asr] @ git+https://github.com/NVIDIA/NeMo.git@main" jiwer

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
import os
import sys

from google.colab import drive

drive.mount("/content/drive")
os.chdir(PROJECT)
sys.path.insert(0, f"{PROJECT}/src")

import nemo
import torch

GPU = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
if BATCH_DURATION is None:
    BATCH_DURATION = 200 if GPU_GB > 70 else 100
print(f"NeMo {nemo.__version__}, {GPU}, {GPU_GB:.0f} GB, batch duration {BATCH_DURATION} s")

Mounted at /content/drive
NeMo 3.1.0+c02239ab4, NVIDIA A100-SXM4-40GB, 39 GB, batch duration 100 s


In [4]:
import logging
import warnings

from nemo.utils import logging as nemo_logging

warnings.filterwarnings("ignore")  # Python warnings: torch, lhotse, Lightning
nemo_logging.setLevel(logging.ERROR)  # NeMo's [NeMo W ...] and [NeMo I ...] lines
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)  # Lightning's GPU and setup messages

## Audio

The zips are copied from Drive to the local disk and unzipped there, because reading 41,000 clips from Drive during training is slow. A split that is already unzipped in this session is skipped.

In [5]:
import shutil
import zipfile
from pathlib import Path

from tqdm.auto import tqdm

for split, zip_path in ZIPS.items():
    target = Path(AUDIO_ROOT) / split
    if target.exists():
        continue
    local_zip = Path("/content") / Path(zip_path).name
    shutil.copy(zip_path, local_zip)
    with zipfile.ZipFile(local_zip) as z:
        for member in tqdm(z.infolist(), desc=f"unzip {split}"):
            z.extract(member, target)
    local_zip.unlink()

unzip train:   0%|          | 0/83064 [00:00<?, ?it/s]

unzip validation:   0%|          | 0/4118 [00:00<?, ?it/s]

unzip test:   0%|          | 0/5886 [00:00<?, ?it/s]

## Manifests

The project manifests have laptop paths, so each clip is found by file name in the unzipped folder. The labels are already normalized and are used as they are. NeMo reads the language prompt from `lang` and `target_lang` in each line.

In [6]:
import json

rows, paths = {}, {}
for split in SPLITS:
    wavs = {p.name: str(p) for p in (Path(AUDIO_ROOT) / split).rglob("*.wav")}
    with open(f"data/processed/openslr104/{split}_manifest.jsonl", encoding="utf-8") as f:
        rows[split] = [json.loads(line) for line in f]
    missing = [r["utt_id"] for r in rows[split] if Path(r["audio_path"]).name not in wavs]
    assert not missing, f"{split}: {len(missing)} clips missing, e.g. {missing[:3]}"
    paths[split] = [wavs[Path(r["audio_path"]).name] for r in rows[split]]
    print(f"{split}: {len(rows[split])} clips, {sum(r['duration'] for r in rows[split]) / 3600:.1f} h")
refs = {split: [r["text"] for r in rows[split]] for split in SPLITS}


def write_manifest(path, split_rows, split_paths):
    with open(path, "w", encoding="utf-8") as f:
        for r, p in zip(split_rows, split_paths):
            line = {"audio_filepath": p, "duration": r["duration"], "text": r["text"], "lang": PROMPT, "target_lang": PROMPT}
            f.write(json.dumps(line, ensure_ascii=False) + "\n")


os.makedirs(MANIFEST_DIR, exist_ok=True)
TRAIN_MANIFEST = f"{MANIFEST_DIR}/train.json"
SMALL_MANIFEST = f"{MANIFEST_DIR}/small.json"
write_manifest(TRAIN_MANIFEST, rows["train"], paths["train"])
write_manifest(SMALL_MANIFEST, rows["train"][:SMALL_N], paths["train"][:SMALL_N])
with open(TRAIN_MANIFEST, encoding="utf-8") as f:
    print(sum(1 for _ in f), "lines in", TRAIN_MANIFEST)
    f.seek(0)
    print(f.readline())

train: 41531 clips, 67.5 h
validation: 2058 clips, 3.4 h
test: 2942 clips, 4.8 h
41531 lines in /content/manifests/train.json
{"audio_filepath": "/content/openslr104/train/train/654444_0049FmqkWQMNW6Tc_0000.wav", "duration": 3.0, "text": "meet the gimp के spoken tutorial में आपका स्वागत है", "lang": "hi-IN", "target_lang": "hi-IN"}



## Scoring

`transcribe` runs whole clips with the attention limited to one streaming latency, then puts the training setting back. `validate` scores the validation set at 560 ms, prints 10 outputs next to their references, and adds a line to `val_scores.csv` on Drive. `state.json` on Drive keeps the best WER, the stop counter, the GPU types and the training hours across sessions.

In [7]:
import csv
import datetime
import re

from hinglish_asr.metrics import score
from hinglish_asr.utils import Normalizer

norm = Normalizer("data/lists/loanwords.csv", "data/lists/transliterations.csv")
assert norm.transliterations, "data/lists/transliterations.csv is missing or empty"
assert score(refs["validation"], refs["validation"], norm) == {"wer": 0.0, "t_wer": 0.0}

LANG_TAG = re.compile(r"\s*<[a-z]{2}-[A-Z]{2}>")
CKPT = Path(CKPT_DIR)
CKPT.mkdir(parents=True, exist_ok=True)
LAST, BEST = CKPT / "last.ckpt", CKPT / "best.nemo"
VAL_LOG, STATE = CKPT / "val_scores.csv", CKPT / "state.json"
EXAMPLES = range(0, len(rows["validation"]), len(rows["validation"]) // 10)[:10]


def transcribe(model, audio_paths, ms, language=PROMPT):
    default, was_training = list(model.encoder.att_context_size), model.training
    model.encoder.set_default_att_context_size(att_context_size=CONTEXT[ms])
    try:
        out = model.transcribe(audio_paths, batch_size=INFER_BATCH, target_lang=language, verbose=True)
    finally:
        model.encoder.set_default_att_context_size(att_context_size=default)
        model.train(was_training)
    out = out[0] if isinstance(out, tuple) else out
    return [LANG_TAG.sub("", h.text if hasattr(h, "text") else h).strip() for h in out]


def validate(model, step):
    hyps = transcribe(model, paths["validation"], VAL_MS)
    s = score(refs["validation"], hyps, norm)
    for i in EXAMPLES:
        print("REF:", refs["validation"][i])
        print("HYP:", hyps[i], "\n")
    new_file = not VAL_LOG.exists()
    with open(VAL_LOG, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if new_file:
            writer.writerow(["step", "wer", "t_wer", "time"])
        writer.writerow([step, round(s["wer"], 4), round(s["t_wer"], 4), datetime.datetime.now().isoformat(timespec="seconds")])
    print(f"step {step}: validation WER {s['wer']:.4f}  T-WER {s['t_wer']:.4f} at {VAL_MS} ms")
    return s


def load_state():
    if STATE.exists():
        return json.loads(STATE.read_text())
    return {"best_wer": None, "best_step": None, "bad_evals": 0, "train_hours": 0.0, "gpus": []}


def save_state(state):
    STATE.write_text(json.dumps(state, indent=1))


print(CKPT, "exists:", sorted(p.name for p in CKPT.iterdir()))

/content/drive/MyDrive/projects/asr/checkpoints/nemotron-3.5-asr-streaming-0.6b exists: ['val_scores.csv']


## Model

`configure` applies only the data, optimizer and trainer settings. The streaming settings are printed before and after, and must not change. Every training batch must carry the `hi-IN` prompt index.

In [8]:
import lightning.pytorch as pl
from omegaconf import OmegaConf

from nemo.collections.asr.models import ASRModel


def load_model():
    model = ASRModel.from_pretrained(MODEL, map_location="cuda")
    assert "Prompt" in type(model).__name__, type(model).__name__
    assert model.cfg.encoder.d_model == D_MODEL, model.cfg.encoder.d_model
    print(type(model).__name__, "| decoding:", model.cfg.decoding.strategy, "| contexts:", model.encoder.att_context_size_all)
    return model


def streaming_settings(model):
    enc = OmegaConf.to_container(model.cfg.encoder, resolve=True)
    return {
        "cfg att_context_size": enc.get("att_context_size"),
        "cfg att_context_probs": enc.get("att_context_probs"),
        "encoder att_context_size_all": model.encoder.att_context_size_all,
        "encoder att_context_probs": model.encoder.att_context_probs,
    }


def data_config(model, manifest, batch_duration, bucketing):
    return OmegaConf.create({
        "manifest_filepath": manifest,
        "sample_rate": model.cfg.sample_rate,
        "use_lhotse": True,
        "batch_duration": batch_duration,
        "use_bucketing": bucketing,
        "num_buckets": NUM_BUCKETS,
        "quadratic_duration": QUADRATIC_DURATION,
        "bucket_buffer_size": 10000,
        "shuffle_buffer_size": 10000,
        "shuffle": True,
        "num_workers": NUM_WORKERS,
        "pin_memory": True,
        "min_duration": 0.1,
        "max_duration": 40.0,
        "prompt_dictionary": model.cfg.model_defaults.prompt_dictionary,
        "num_prompts": model.cfg.model_defaults.num_prompts,
        "subsampling_factor": model.cfg.encoder.subsampling_factor,
        "lang_field": "target_lang",
        "default_prompt_mode": "langID",  # every clip gets PROMPT, never the random auto prompt
    })


def configure(model, trainer, manifest, batch_duration, bucketing=True):
    before = streaming_settings(model)
    model.set_trainer(trainer)
    model.setup_training_data(data_config(model, manifest, batch_duration, bucketing))
    model.setup_optimization(OmegaConf.create(OPTIM))
    after = streaming_settings(model)
    print("streaming before:", before)
    print("streaming after: ", after)
    assert before == after, "the training config changed the streaming settings"

    batch = next(iter(model._train_dl))
    prompt_index = model.cfg.model_defaults.prompt_dictionary[PROMPT]
    assert len(batch) == 5 and (batch[4] == prompt_index).all(), batch[4]
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    assert trainable == total, (trainable, total)
    print(f"{PROMPT} prompt index {prompt_index} on every clip of a {len(batch[0])}-clip batch; {trainable / 1e6:.1f}M of {total / 1e6:.1f}M parameters train")
    print(f"peak learning rate {OPTIM['lr'] / (D_MODEL * WARMUP_STEPS) ** 0.5:.1e} at step {WARMUP_STEPS}")


def make_trainer(max_steps, callbacks):
    return pl.Trainer(
        devices=1,
        accelerator="gpu",
        precision=PRECISION,
        max_steps=max_steps,
        max_epochs=-1,
        gradient_clip_val=GRAD_CLIP,
        use_distributed_sampler=False,  # lhotse does its own sampling
        limit_val_batches=0,  # validation runs in the callback with the project score
        num_sanity_val_steps=0,
        logger=False,
        enable_checkpointing=False,  # checkpoints are written by the callback
        callbacks=callbacks,
    )


model = load_model()

nemotron-3.5-asr-streaming-0.6b.nemo: reconstructing file:   0%|          |  0.00B / 2.37GB            

nemotron-3.5-asr-streaming-0.6b.nemo: downloading bytes:           |  0.00B            

EncDecRNNTBPEModelWithPrompt | decoding: greedy_batch | contexts: [[56, 3], [56, 0], [56, 6], [56, 13]]


## Tokenizer

Each distinct label word is encoded and decoded again. A character that is lost in that round trip cannot be encoded.

In [9]:
from collections import Counter

words = Counter(w for split in ("train", "validation") for t in refs[split] for w in t.split())
lost = Counter()
for w, n in words.items():
    decoded = model.tokenizer.ids_to_text(model.tokenizer.text_to_ids(w)).strip()
    if decoded != w:
        for ch in set(w) - set(decoded):
            lost[ch] += n * w.count(ch)
total_chars = sum(n * len(w) for w, n in words.items())
print(f"{sum(lost.values())} of {total_chars} label characters cannot be encoded ({len(lost)} distinct)")
print(lost.most_common(20))

8 of 2133100 label characters cannot be encoded (1 distinct)
[('ऩ', 8)]


## Check: step 0 on the validation set

In [10]:
if VAL_LOG.exists():
    print(VAL_LOG.read_text())
else:
    validate(model, step=0)

step,wer,t_wer,time
0,0.4798,0.2972,2026-11-23T06:28:36



## Check: small set

The model trains on 50 clips for a few minutes. The loss must fall close to zero (below 10% of its start; SpecAugment stays on, so it does not reach exactly zero). Nothing is saved, and the base model is loaded again afterwards.

In [11]:
import gc
import time


class LossLog(pl.Callback):
    def __init__(self):
        self.losses = []

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        self.losses.append(float(outputs["loss"]))


log = LossLog()
small_trainer = make_trainer(SMALL_STEPS, [log])
configure(model, small_trainer, SMALL_MANIFEST, SMALL_BATCH_DURATION, bucketing=False)
start = time.time()
small_trainer.fit(model)
first, last = sum(log.losses[:5]) / 5, sum(log.losses[-5:]) / 5
print(f"{len(log.losses)} steps in {(time.time() - start) / 60:.1f} min: loss {first:.3f} -> {last:.3f}")
assert last < 0.1 * first

del model, small_trainer
gc.collect()
torch.cuda.empty_cache()
model = load_model()

INFO:pytorch_lightning.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


streaming before: {'cfg att_context_size': [[56, 3], [56, 0], [56, 6], [56, 13]], 'cfg att_context_probs': None, 'encoder att_context_size_all': [[56, 3], [56, 0], [56, 6], [56, 13]], 'encoder att_context_probs': [0.25, 0.25, 0.25, 0.25]}
streaming after:  {'cfg att_context_size': [[56, 3], [56, 0], [56, 6], [56, 13]], 'cfg att_context_probs': None, 'encoder att_context_size_all': [[56, 3], [56, 0], [56, 6], [56, 13]], 'encoder att_context_probs': [0.25, 0.25, 0.25, 0.25]}
hi-IN prompt index 6 on every clip of a 5-clip batch; 638.0M of 638.0M parameters train
peak learning rate 1.0e-04 at step 100


INFO:pytorch_lightning.utilities.rank_zero:You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=200` reached.


200 steps in 3.6 min: loss 123.460 -> 9.687
EncDecRNNTBPEModelWithPrompt | decoding: greedy_batch | contexts: [[56, 3], [56, 0], [56, 6], [56, 13]]


## Train

The callback prints the loss every 100 steps, scores the validation set every `VAL_EVERY` steps, saves `best.nemo` when the WER is the lowest so far, and stops after `PATIENCE` runs without a lower WER. It writes `last.ckpt` every 30 minutes and after each validation (to a temporary file first, so a disconnect cannot leave a broken checkpoint). Training continues from `last.ckpt` when it exists.

In [12]:
class TrainCallback(pl.Callback):
    def __init__(self, state):
        self.state = state
        self.clock = self.saved = time.time()

    def add_hours(self):
        now = time.time()
        self.state["train_hours"] += (now - self.clock) / 3600
        self.clock = now
        save_state(self.state)

    def save_last(self, trainer):
        tmp = LAST.with_suffix(".tmp")
        trainer.save_checkpoint(str(tmp))
        os.replace(tmp, LAST)
        self.saved = time.time()

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        step = trainer.global_step
        if step % 100 == 0:
            lr = trainer.optimizers[0].param_groups[0]["lr"]
            print(f"step {step}: loss {float(outputs['loss']):.3f}, lr {lr:.2e}, peak memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")
        if step % VAL_EVERY == 0:
            s = validate(pl_module, step)
            if self.state["best_wer"] is None or s["wer"] < self.state["best_wer"]:
                self.state.update(best_wer=s["wer"], best_step=step, bad_evals=0)
                pl_module.save_to(str(BEST))
                print("new best, saved", BEST)
            else:
                self.state["bad_evals"] += 1
                print(f"no improvement ({self.state['bad_evals']} of {PATIENCE})")
                trainer.should_stop = self.state["bad_evals"] >= PATIENCE
            self.add_hours()
            self.save_last(trainer)
        elif time.time() - self.saved > CKPT_MINUTES * 60:
            self.add_hours()
            self.save_last(trainer)

    def on_train_end(self, trainer, pl_module):
        self.add_hours()


state = load_state()
if GPU not in state["gpus"]:
    state["gpus"].append(GPU)
save_state(state)

if state["bad_evals"] >= PATIENCE:
    print(f"Training already stopped at step {state['best_step']} (best WER {state['best_wer']:.4f}).")
else:
    trainer = make_trainer(MAX_STEPS, [TrainCallback(state)])
    configure(model, trainer, TRAIN_MANIFEST, BATCH_DURATION)
    print("continuing from", LAST if LAST.exists() else "the base model")
    trainer.fit(model, ckpt_path=str(LAST) if LAST.exists() else None)

INFO:pytorch_lightning.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


streaming before: {'cfg att_context_size': [[56, 3], [56, 0], [56, 6], [56, 13]], 'cfg att_context_probs': None, 'encoder att_context_size_all': [[56, 3], [56, 0], [56, 6], [56, 13]], 'encoder att_context_probs': [0.25, 0.25, 0.25, 0.25]}
streaming after:  {'cfg att_context_size': [[56, 3], [56, 0], [56, 6], [56, 13]], 'cfg att_context_probs': None, 'encoder att_context_size_all': [[56, 3], [56, 0], [56, 6], [56, 13]], 'encoder att_context_probs': [0.25, 0.25, 0.25, 0.25]}
hi-IN prompt index 6 on every clip of a 2-clip batch; 638.0M of 638.0M parameters train
peak learning rate 1.0e-04 at step 100
continuing from the base model


Training: |          | 0/? [00:00<?, ?it/s]

step 100: loss 44.905, lr 1.00e-04, peak memory 20.7 GB
step 200: loss 33.374, lr 7.07e-05, peak memory 20.7 GB
step 300: loss 42.019, lr 5.77e-05, peak memory 20.7 GB
step 400: loss 71.725, lr 5.00e-05, peak memory 20.7 GB
step 500: loss 103.441, lr 4.47e-05, peak memory 20.7 GB
step 600: loss 61.474, lr 4.08e-05, peak memory 20.7 GB
step 700: loss 39.012, lr 3.78e-05, peak memory 20.7 GB
step 800: loss 17.020, lr 3.54e-05, peak memory 20.7 GB
step 900: loss 17.770, lr 3.33e-05, peak memory 20.7 GB
step 1000: loss 25.197, lr 3.16e-05, peak memory 20.7 GB


[NeMo W 2026-11-23 06:50:33 dataloader:1308] The following configuration keys are ignored by Lhotse dataloader: initialize_prompt_feature,num_prompts,window_stride,default_prompt_mode,labels,default_lang,subsampling_factor,target_lang,prompt_dictionary,trim_silence
[NeMo W 2026-11-23 06:50:33 dataloader:883] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)

Transcribing: 0it [00:00, ?it/s]
Transcribing: 1it [00:23, 23.22s/it]
Transcribing: 2it [00:23,  9.97s/it]
Transcribing: 3it [00:24,  5.76s/it]
Transcribing: 4it [00:26,  4.02s/it]
Transcribing: 5it [00:27,  2.95s/it]
Transcribing: 6it [00:28,  2.36s/it]
Transcribing: 7it [00:29,  2.05s/it]
Trans

REF: नमस्कार दोस्तों
HYP: नमस्कार दोस्तों 

REF: मैं इसे एक बार और दोहराता हूँ
HYP: मैं इसे एक बार और दोहराता हूँ 

REF: content पर click करें और सूची में user group पर click करें
HYP: content पर click करें और सूची में user group पर click करें 

REF: उदाहरणस्वरूप मैं content चुनूँगी path pattern field में हमें pattern template प्रविष्ट करना होगा
HYP: उदाहरणस्वरूप मैं content चुनूँगी path pattern field में हमें pattern template प्रविष्ट करना होगा 

REF: ध्यान दें कि curve का परिवर्तन smooth है
HYP: ध्यान दें कि curve का परिवर्तन smooth है 

REF: यदि value character या string है तो variable को double quotes में value के साथ आरंभ किया जाता है
HYP: यदि value character या string है तो variable को double quotes में value के साथ आरंभ किया जाता है 

REF: author name file delete करें
HYP: delete करें 

REF: education और research के लिए डिज़ाइन किया गया है
HYP: education और research के लिए डिजाइन किया गया है 

REF: अब सारांशित करते हैं कि हमने क्या सीखा
HYP: अब सारांशित करते हैं कि हमने क्या सीख

... (6 outputs left out)

[NeMo W 2026-11-23 07:28:25 dataloader:1308] The following configuration keys are ignored by Lhotse dataloader: initialize_prompt_feature,num_prompts,window_stride,default_prompt_mode,labels,default_lang,subsampling_factor,target_lang,prompt_dictionary,trim_silence
[NeMo W 2026-11-23 07:28:25 dataloader:883] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)

Transcribing: 0it [00:00, ?it/s]
Transcribing: 1it [00:00,  1.31it/s]
Transcribing: 2it [00:01,  1.39it/s]
Transcribing: 3it [00:02,  1.38it/s]
Transcribing: 4it [00:03,  1.03it/s]
Transcribing: 5it [00:04,  1.01s/it]
Transcribing: 6it [00:05,  1.09s/it]
Transcribing: 7it [00:07,  1.18s/it]
Trans

REF: नमस्कार दोस्तों
HYP: नमस्कार दोस्तों 

REF: मैं इसे एक बार और दोहराता हूँ
HYP: मैं इसे एक बार और दोहराता हूँ 

REF: content पर click करें और सूची में user group पर click करें
HYP: content पर click करें और सूची में user group पर click करें 

REF: उदाहरणस्वरूप मैं content चुनूँगी path pattern field में हमें pattern template प्रविष्ट करना होगा
HYP: उदाहरणस्वरूप मैं content चुनूँगी path pattern field में हमें pattern template प्रविष्ट करना होगा 

REF: ध्यान दें कि curve का परिवर्तन smooth है
HYP: ध्यान दें कि curve का परिवर्तन smooth है 

REF: यदि value character या string है तो variable को double quotes में value के साथ आरंभ किया जाता है
HYP: यदि value character या string है तो variable को double quotes में value के साथ आरंभ किया जाता है 

REF: author name file delete करें
HYP: delete करें 

REF: education और research के लिए डिज़ाइन किया गया है
HYP: education और research के लिए डिजाइन किया गया है 

REF: अब सारांशित करते हैं कि हमने क्या सीखा
HYP: अब सारांशित करते हैं कि हमने क्या सीख

[NeMo W 2026-11-23 07:36:52 dataloader:1308] The following configuration keys are ignored by Lhotse dataloader: initialize_prompt_feature,num_prompts,window_stride,default_prompt_mode,labels,default_lang,subsampling_factor,target_lang,prompt_dictionary,trim_silence
[NeMo W 2026-11-23 07:36:52 dataloader:883] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)

Transcribing: 0it [00:00, ?it/s]
Transcribing: 1it [00:00,  1.38it/s]
Transcribing: 2it [00:01,  1.41it/s]
Transcribing: 3it [00:02,  1.39it/s]
Transcribing: 4it [00:03,  1.02it/s]
Transcribing: 5it [00:04,  1.01s/it]
Transcribing: 6it [00:05,  1.07s/it]
Transcribing: 7it [00:07,  1.20s/it]
Trans

REF: नमस्कार दोस्तों
HYP: नमस्कार दोस्तों 

REF: मैं इसे एक बार और दोहराता हूँ
HYP: मैं इसे एक बार और दोहराता हूँ 

REF: content पर click करें और सूची में user group पर click करें
HYP: content पर click करें और सूची में user group पर click करें 

REF: उदाहरणस्वरूप मैं content चुनूँगी path pattern field में हमें pattern template प्रविष्ट करना होगा
HYP: उदाहरणस्वरूप मैं content चुनूँगी path pattern field में हमें pattern template प्रविष्ट करना होगा 

REF: ध्यान दें कि curve का परिवर्तन smooth है
HYP: ध्यान दें कि curve का परिवर्तन smooth है 

REF: यदि value character या string है तो variable को double quotes में value के साथ आरंभ किया जाता है
HYP: यदि value character या string है तो variable को double quotes में value के साथ आरंभ किया जाता है 

REF: author name file delete करें
HYP: delete करें 

REF: education और research के लिए डिज़ाइन किया गया है
HYP: education और research के लिए डिजाइन किया गया है 

REF: अब सारांशित करते हैं कि हमने क्या सीखा
HYP: अब सारांशित करते हैं कि हमने क्या सीख

## Cost

In [13]:
state = load_state()
print(f"GPU: {', '.join(state['gpus'])}")
print(f"Training time: {state['train_hours']:.2f} h")
print(f"Best validation WER {state['best_wer']:.4f} at step {state['best_step']}")
print(VAL_LOG.read_text())

GPU: NVIDIA A100-SXM4-40GB
Training time: 0.91 h
Best validation WER 0.2235 at step 3000
step,wer,t_wer,time
0,0.4798,0.2972,2026-11-23T06:28:36
1000,0.2458,0.2448,2026-11-23T06:51:51
2000,0.2256,0.225,2026-11-23T07:00:45
3000,0.2235,0.2229,2026-11-23T07:10:29
4000,0.2263,0.2257,2026-11-23T07:19:44
5000,0.2352,0.2347,2026-11-23T07:29:18
6000,0.2272,0.2265,2026-11-23T07:37:45



## Final scores

The best checkpoint is scored on the test set at 560 ms and 1.12 s. Each run's outputs go to `results/nemotron-3.5-asr-streaming-0.6b-finetuned/hyps-<latency>ms.jsonl`, and one row per latency goes to `results/summary.csv` in the same format as the zero-shot rows.

In [14]:
best = ASRModel.restore_from(str(BEST), map_location="cuda")
out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

scores = {}
for ms in CONTEXT:
    hyps = transcribe(best, paths["test"], ms)
    with open(out_dir / f"hyps-{ms}ms.jsonl", "w", encoding="utf-8") as f:
        for r, h in zip(rows["test"], hyps):
            f.write(json.dumps({"utt_id": r["utt_id"], "ref": r["text"], "hyp": h}, ensure_ascii=False) + "\n")
    scores[ms] = score(refs["test"], hyps, norm)
    assert scores[ms]["t_wer"] <= scores[ms]["wer"]
    print(f"{ms} ms: WER {scores[ms]['wer']:.4f}  T-WER {scores[ms]['t_wer']:.4f}, {sum(not h for h in hyps)} empty")

summary = Path("results/summary.csv")
with open(summary, encoding="utf-8") as f:
    fields = next(csv.reader(f))
with open(summary, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    for ms, s in scores.items():
        writer.writerow({
            "model": MODEL,
            "dataset": "openslr104",
            "split": "test",
            "n_utts": len(rows["test"]),
            "wer": round(s["wer"], 4),
            "t_wer": round(s["t_wer"], 4),
            "decoding": (
                f"fine-tuned on OpenSLR 104 train (best checkpoint at step {state['best_step']}, peak lr {PEAK_LR}, batch {BATCH_DURATION} s), "
                f"{best.cfg.decoding.strategy} RNN-T, streaming latency {ms} ms, language={PROMPT}, float32, whole clips"
            ),
            "date": datetime.date.today().isoformat(),
            "scoring": "v2",
        })
print(summary.read_text(encoding="utf-8"))

[NeMo W 2026-11-23 07:39:48 dataloader:1308] The following configuration keys are ignored by Lhotse dataloader: initialize_prompt_feature,num_prompts,window_stride,default_prompt_mode,labels,default_lang,subsampling_factor,target_lang,prompt_dictionary,trim_silence
[NeMo W 2026-11-23 07:39:48 dataloader:883] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 92it [00:58,  1.57it/s]


560 ms: WER 0.3918  T-WER 0.3897, 220 empty


[NeMo W 2026-11-23 07:40:47 dataloader:1308] The following configuration keys are ignored by Lhotse dataloader: initialize_prompt_feature,num_prompts,window_stride,default_prompt_mode,labels,default_lang,subsampling_factor,target_lang,prompt_dictionary,trim_silence
[NeMo W 2026-11-23 07:40:47 dataloader:883] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 92it [00:53,  1.73it/s]


1120 ms: WER 0.3765  T-WER 0.3745, 308 empty
model,dataset,split,n_utts,wer,t_wer,decoding,date,scoring
openai/whisper-small,openslr104,test,2942,0.915,0.915,"zero-shot, temperature fallback 0.0-1.0 (compression ratio 1.35, logprob -1.0, seed 0), language=hi, fp16, 30 s chunks",2026-11-23,v1
Trelis/tara,openslr104,test,2942,0.295,0.295,"zero-shot, temperature fallback 0.0-1.0 (compression ratio 1.35, logprob -1.0, seed 0), language=hi + mixedcode, bf16, 30 s pieces",2026-11-23,v1
Qwen/Qwen3-ASR-0.6B,openslr104,test,2942,0.4362,0.4362,"zero-shot, qwen-asr 0.0.6 defaults (greedy, max 512 new tokens, repetition clean-up), language=None, bf16, whole clips",2026-11-23,v1
nvidia/nemotron-3.5-asr-streaming-0.6b,openslr104,test,2942,0.4584,0.4584,"zero-shot, greedy RNN-T, streaming latency 560 ms, language=auto, float32, whole clips",2026-11-23,v1
nvidia/nemotron-3.5-asr-streaming-0.6b,openslr104,test,2942,0.4546,0.4546,"zero-shot, greedy RNN-T, streaming latency 1120 ms, language=auto, float3